Tâche 2: Extension multi-sujets L’analyse initiale portait uniquement sur le participant sub-005. Je vais adapter le notebook afin d’appliquer l'analyse aux cinq participants (sub-001 à sub-005).

Vérification que chaque sujet à un fichier BOLD (_bold.nii) et des fichiers d'events (_events.tsv)

In [7]:
import pandas as pd
from pathlib import Path

base_path = Path("./ds003720")   # adapte ici
task = "Test"

def run_from_name(path):
    name = path.name
    if "run-" in name:
        idx = name.index("run-") + 4
        return name[idx:idx+2]
    return None

subjects = sorted([p.name.replace("sub-", "") for p in base_path.iterdir()
                   if p.is_dir() and p.name.startswith("sub-")])

rows = []

for sub in subjects:
    subdir = base_path / f"sub-{sub}"

    bold_files = list(subdir.rglob(f"*task-{task}*bold*.nii")) + list(subdir.rglob(f"*task-{task}*bold*.nii.gz"))
    event_files = list(subdir.rglob(f"*task-{task}*events.tsv"))

    bold_runs = {run_from_name(p) for p in bold_files if run_from_name(p) is not None}
    event_runs = {run_from_name(p) for p in event_files if run_from_name(p) is not None}

    all_runs = sorted(bold_runs | event_runs)

    for run in all_runs:
        has_bold = run in bold_runs
        has_events = run in event_runs

        if has_bold and has_events:
            status = "OK"
        elif has_bold and not has_events:
            status = "MISSING_EVENTS"
        elif not has_bold and has_events:
            status = "MISSING_BOLD"
        else:
            status = "MISSING_BOTH"

        rows.append({
            "subject": f"sub-{sub}",
            "run": run,
            "bold_present": has_bold,
            "events_present": has_events,
            "status": status
        })

report = pd.DataFrame(rows).sort_values(["subject", "run"]).reset_index(drop=True)

report.to_csv("bold_events_presence_report.csv", index=False)
print(report)
print("CSV saved: bold_events_presence_report.csv")

    subject run  bold_present  events_present status
0   sub-001  01          True            True     OK
1   sub-001  02          True            True     OK
2   sub-001  03          True            True     OK
3   sub-001  04          True            True     OK
4   sub-001  05          True            True     OK
5   sub-001  06          True            True     OK
6   sub-002  01          True            True     OK
7   sub-002  02          True            True     OK
8   sub-002  03          True            True     OK
9   sub-002  04          True            True     OK
10  sub-002  05          True            True     OK
11  sub-002  06          True            True     OK
12  sub-003  01          True            True     OK
13  sub-003  02          True            True     OK
14  sub-003  03          True            True     OK
15  sub-003  04          True            True     OK
16  sub-003  05          True            True     OK
17  sub-003  06          True            True 

Vérification de la structure des fichiers events

In [9]:
import pandas as pd
from pathlib import Path

base_path = Path("./ds003720")  # adapte ici
task = "Test"

rows = []

for subdir in sorted(base_path.glob("sub-*")):
    for f in subdir.rglob(f"*task-{task}*events.tsv"):
        df = pd.read_csv(f, sep="\t")
        rows.append({
            "subject": subdir.name,
            "file": f.name,
            "n_rows": len(df),
            "columns": "|".join(df.columns.astype(str)),
            "has_onset": "onset" in df.columns,
            "has_duration": "duration" in df.columns,
            "has_genre": "genre" in df.columns,
            "has_track": "track" in df.columns,
            "has_start": "start" in df.columns,
            "has_end": "end" in df.columns,
        })

structure_df = pd.DataFrame(rows)
structure_df.to_csv("events_structure_report.csv", index=False)
print(structure_df)

    subject                                 file  n_rows  \
0   sub-001  sub-001_task-Test_run-01_events.tsv      41   
1   sub-001  sub-001_task-Test_run-02_events.tsv      41   
2   sub-001  sub-001_task-Test_run-04_events.tsv      41   
3   sub-001  sub-001_task-Test_run-06_events.tsv      41   
4   sub-001  sub-001_task-Test_run-03_events.tsv      41   
5   sub-001  sub-001_task-Test_run-05_events.tsv      41   
6   sub-002  sub-002_task-Test_run-01_events.tsv      41   
7   sub-002  sub-002_task-Test_run-06_events.tsv      41   
8   sub-002  sub-002_task-Test_run-02_events.tsv      41   
9   sub-002  sub-002_task-Test_run-05_events.tsv      41   
10  sub-002  sub-002_task-Test_run-03_events.tsv      41   
11  sub-002  sub-002_task-Test_run-04_events.tsv      41   
12  sub-003  sub-003_task-Test_run-01_events.tsv      41   
13  sub-003  sub-003_task-Test_run-06_events.tsv      41   
14  sub-003  sub-003_task-Test_run-05_events.tsv      41   
15  sub-003  sub-003_task-Test_run-02_ev

In [12]:
import pandas as pd
from pathlib import Path

base_path = Path("./ds003720")  # adapte ici
task = "Test"

rows = []

for f in sorted(base_path.glob(f"sub-*/**/*task-{task}*events.tsv")):
    df = pd.read_csv(f, sep="\t")

    sub = [p for p in f.parts if p.startswith("sub-")][0]
    run = None
    if "run-" in f.name:
        run = f.name.split("run-")[1][:2]

    rows.append({
        "subject": sub,
        "run": run,
        "file": str(f),
        "n_rows": len(df),
        "columns": ", ".join(df.columns.astype(str)),
        "head_10": df.head(10).to_csv(index=False).replace("\n", " | "),
        "genre_counts": df["genre"].value_counts().to_dict() if "genre" in df.columns else None,
        "onset_min": float(df["onset"].min()) if "onset" in df.columns and len(df) else None,
        "onset_max": float(df["onset"].max()) if "onset" in df.columns and len(df) else None,
        "duration_unique": sorted(df["duration"].dropna().astype(float).round(3).unique().tolist())[:20] if "duration" in df.columns else None,
    })

summary = pd.DataFrame(rows)
summary.to_csv("events_content_summary.csv", index=False)

print(summary[["subject", "run", "n_rows", "columns"]].head(20).to_string(index=False))
print("Saved: events_content_summary.csv")

subject run  n_rows                                   columns
sub-001  01      41 onset, duration, genre, track, start, end
sub-001  02      41 onset, duration, genre, track, start, end
sub-001  03      41 onset, duration, genre, track, start, end
sub-001  04      41 onset, duration, genre, track, start, end
sub-001  05      41 onset, duration, genre, track, start, end
sub-001  06      41 onset, duration, genre, track, start, end
sub-002  01      41 onset, duration, genre, track, start, end
sub-002  02      41 onset, duration, genre, track, start, end
sub-002  03      41 onset, duration, genre, track, start, end
sub-002  04      41 onset, duration, genre, track, start, end
sub-002  05      41 onset, duration, genre, track, start, end
sub-002  06      41 onset, duration, genre, track, start, end
sub-003  01      41 onset, duration, genre, track, start, end
sub-003  02      41 onset, duration, genre, track, start, end
sub-003  03      41 onset, duration, genre, track, start, end
sub-003 

In [15]:
import pandas as pd
from pathlib import Path

base_path = Path("./ds003720")  # adapte ici
task = "Test"
required_cols = ["onset", "duration", "genre", "track", "start", "end"]

all_rows = []
missing_rows = []

for f in sorted(base_path.glob(f"sub-*/**/*task-{task}*events.tsv")):
    df = pd.read_csv(f, sep="\t")
    sub = [p for p in f.parts if p.startswith("sub-")][0]
    run = None
    if "run-" in f.name:
        run = f.name.split("run-")[1][:2]

    missing = [c for c in required_cols if c not in df.columns]

    missing_rows.append({
        "subject": sub,
        "file": str(f),
        "missing_columns": ", ".join(missing) if missing else "",
        "n_trials": len(df),
        "genres_present": ", ".join(sorted(df["genre"].dropna().astype(str).unique())) if "genre" in df.columns else ""
    })

    keep_cols = [c for c in required_cols if c in df.columns]
    tmp = df[keep_cols].copy()
    tmp.insert(0, "subject", sub)
    tmp.insert(1, "run", run)
    all_rows.append(tmp)

global_df = pd.concat(all_rows, ignore_index=True)
global_df.to_csv("events_global_table.csv", index=False)

subject_summary = global_df.groupby("subject").agg(
    n_trials=("subject", "size"),
    n_runs=("run", "nunique")
).reset_index()

subject_summary["genres_present"] = subject_summary["subject"].map(
    lambda s: ", ".join(sorted(global_df.loc[global_df["subject"] == s, "genre"].dropna().astype(str).unique()))
    if "genre" in global_df.columns else ""
)

subject_summary.to_csv("events_subject_summary.csv", index=False)

missing_df = pd.DataFrame(missing_rows)
missing_df.to_csv("events_missing_columns_report.csv", index=False)

print("Saved:")
print("- events_global_table.csv")
print("- events_subject_summary.csv")
print("- events_missing_columns_report.csv")

Saved:
- events_global_table.csv
- events_subject_summary.csv
- events_missing_columns_report.csv


À la suite de la vérification des fichiers events et bold, tous les sujets disposent bien de leurs fichiers correspondants, et l’ensemble des fichiers est aligné correctement